In [1]:
import pandas as pd
import ast
import re

In [3]:
# ── Config ──────────────────────────────────────────────────────────────────
CSV_PATH = 'C:/Users/rrkar/Downloads/duolingo_vocab_tata_16052026_2026-05-19_2130.csv'
nativeLanguage = 'pl'   # change as needed  (e.g. 'pl', 'es')
targetLanguage = 'en'   # change as needed

LB_FW = {
  'en': {
    'the','a','an','in','on','at','by','for','with','about','to','of','from',
    'into','onto','upon','over','under','above','below','between','among',
    'through','during','before','after','against','along','around','behind',
    'beside','besides','beyond','down','except','inside','near','off','out',
    'outside','past','since','throughout','till','toward','towards','underneath',
    'until','up','via','within','without','and','but','or','nor','so','yet',
    'both','either','neither','whether','although','because','since','unless',
    'until','while','whereas','though','even','if','when','where','as','that',
    'than','then','once','now','provided','lest',
    'i','me','my','mine','myself','you','your','yours','yourself','yourselves',
    'he','him','his','himself','she','her','hers','herself','it','its','itself',
    'we','us','our','ours','ourselves','they','them','their','theirs','themselves',
    'who','whom','whose','which','what','this','that','these','those','one','ones',
    'someone','anyone','everyone','nobody','somebody','anybody','everybody',
    'something','anything','everything','nothing','each','another','other','others',
    'be','am','is','are','was','were','been','being','have','has','had','having',
    'do','does','did','done','doing','will','would','shall','should','may','might',
    'must','can','could','need','dare','ought','get','got','gotten',
    'some','any','all','much','many','more','most','few','fewer','little','less',
    'least','several','enough','such','same','last','next','own',
    'not','also','just','only','even','still','yet','already','always','never',
    'often','sometimes','usually','here','there','now','then','very','too','so',
    'rather','quite','almost','nearly','hardly','barely','merely','indeed',
    'perhaps','maybe','however','therefore','thus','hence','otherwise','anyway',
    'instead','else','together','apart','away','back','forward','again','once',
    'twice','well','like',
  },
  'pl': {
    'w','we','na','do','z','ze','dla','przez','przy','po','przed','nad','pod',
    'za','między','o','od','ku','mimo','według','wokół','poza','obok',
    'naprzeciwko','spod','spośród','sprzed','znad','zza','bez','wzdłuż',
    'podczas','wobec','odnośnie','dokoła','wokoło','wewnątrz','zewnątrz',
    'i','a','ale','lecz','lub','albo','ani','bo','że','czy','jak','kiedy','gdy',
    'jeśli','jeżeli','chociaż','choć','skoro','ponieważ','gdyż','dlatego','więc',
    'zatem','jednak','toteż','żeby','aby','ażeby','dopóki','dopóty','zanim',
    'odkąd','choćby','byle','byleby',
    'ja','ty','on','ona','ono','my','wy','oni','one','mnie','mi','mną',
    'ciebie','cię','tobie','ci','tobą','go','jego','jemu','mu','nim','jej',
    'ją','nią','nas','nam','nami','was','wam','wami','ich','im','nimi','się',
    'sobie','siebie','ten','ta','to','ci','te','tego','tej','temu','tym','tą',
    'tych','tymi','tamten','tamta','tamto','tamci','tamte','kto','co','który',
    'która','które','komu','czemu','kogo','czego','kogoś','coś','ktoś','nic',
    'nikt','wszystko','wszyscy','każdy','każda','każde','żaden','żadna','żadne',
    'być','jest','są','był','była','było','byli','były','będzie','będą','będę',
    'będziesz','będziemy','będziecie','byłem','byłam','byłeś','byłaś','byliśmy',
    'byłyśmy','byliście','byłyście','mieć','ma','mają','mam','masz','mamy',
    'macie','miał','miała','miało','mieli','miały','móc','mogę','możesz','może',
    'możemy','możecie','mogą','mógł','mogła',
    'jakiś','jakaś','jakieś','każdy','każda','każde','wszystek','cały','całe',
    'całą','parę','kilka','wiele','mało','dużo','więcej','mniej','trochę',
    'nieco','tyle','ile','kilku','kilkoma',
    'nie','też','już','jeszcze','zawsze','nigdy','często','rzadko','czasem',
    'zwykle','tutaj','tam','tu','teraz','wtedy','potem','bardzo','tylko',
    'nawet','właśnie','raczej','prawie','wcale','chyba','może','natomiast',
    'tymczasem','przeto','stąd','wszędzie','nigdzie','tak',
    'jeden','jedna','jedno','dwa','dwie','trzy',
    'by','niby','bodaj','byle','chyba','chociażby','akurat',
  },
  'es': {
    'el','la','los','las','un','una','unos','unas','lo',
    'en','de','a','con','por','para','sin','sobre','entre','hasta','desde',
    'ante','bajo','cabe','contra','durante','hacia','mediante','salvo','según',
    'so','tras','versus','vía',
    'y','e','o','u','ni','pero','sino','mas','aunque','porque','pues','que',
    'como','cuando','si','donde','mientras',
    'yo','tú','él','ella','ello','nosotros','nosotras','vosotros','vosotras',
    'ellos','ellas','usted','ustedes','me','te','se','nos','os','le','les',
    'lo','la','los','las','mí','ti','sí','mi','tu','su','nuestro','nuestra',
    'nuestros','nuestras','vuestro','vuestra','vuestros','vuestras',
    'este','esta','esto','estos','estas','ese','esa','eso','esos','esas',
    'aquel','aquella','aquello','aquellos','aquellas','quien','quienes','cual',
    'cuales','cuyo','cuya','cuyos','cuyas','alguien','nadie','algo','nada',
    'todo','todos','todas','cada','cualquier',
    'ser','soy','eres','es','somos','sois','son','era','eras','éramos','erais',
    'eran','fui','fuiste','fue','fuimos','fuisteis','fueron','sido','siendo',
    'estar','estoy','estás','está','estamos','estáis','están','estaba','estuvo',
    'estuvimos','estuvieron','estado','estando','haber','he','has','ha','hemos',
    'habéis','han','había','hubo','habido','tener','tengo','tienes','tiene',
    'tenemos','tenéis','tienen','poder','puedo','puedes','puede','podemos',
    'podéis','pueden','deber','querer','saber','hacer','ir','voy','vas','va',
    'vamos','vais','van',
    'algún','alguna','ningún','ninguna','otro','otra','otros','otras','mucho',
    'mucha','muchos','muchas','poco','poca','pocos','pocas','toda','tanto',
    'tanta','tantos','tantas','más','menos','bastante','demasiado','varios',
    'varias','mismo','misma',
    'no','sí','también','tampoco','ya','aún','todavía','siempre','nunca',
    'jamás','aquí','ahí','allí','acá','allá','ahora','entonces','después',
    'antes','muy','bien','mal','así','tan','solo','sólo','incluso','hasta',
    'además','quizás','quizá','tal vez','acaso',
  },
}

target_fnwords = LB_FW.get(targetLanguage, set())
native_fnwords = LB_FW.get(nativeLanguage, set())

def parse_cell(cell):
    if pd.isna(cell) or str(cell).strip() == '':
        return []
    return [word.strip() for word in str(cell).strip().split(',') if word.strip()]

def is_fnword(word, fnword_set):
    return word.lower().strip() in fnword_set

# ── Load ─────────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH, header=0, delimiter=';')
df.columns = ['target_word', 'translations', 'forms', 'source']

original_len = len(df)

# ── Step 1: Drop rows where col1 is a target function word ───────────────────
mask_fnword_col1 = df['target_word'].apply(lambda w: is_fnword(str(w), target_fnwords))
df = df[~mask_fnword_col1].copy()

# ── Step 2 & 3: Filter col2 translations; drop row if list becomes empty ─────

def filter_cell(cell):
    words = parse_cell(cell)
    return [word for word in words if not is_fnword(word, native_fnwords)]

df['translations_list'] = df['translations'].apply(filter_cell)
df['forms_list'] = df['forms'].apply(filter_cell)

df = df[df['translations_list'].apply(lambda x: len(x) > 0)].copy()

# ── Reconstruct clean string columns ─────────────────────────────────────────
df['translations'] = df['translations_list'].apply(lambda x: str(x) if x else '[]')
df['forms']        = df['forms_list'].apply(lambda x: str(x) if x else '[]')
df = df.drop(columns=['translations_list', 'forms_list'])



# ── Step 5: Report ────────────────────────────────────────────────────────────
print(f"Original rows       : {original_len}")
print(f"Remaining rows      : {len(df)}")

# ── Preview ───────────────────────────────────────────────────────────────────
df.head()

Original rows       : 1293
Remaining rows      : 1146


,target_word,translations,forms,source
0,fast food,['fast food'],[],duolingo
1,stressed out,['zestresowany'],[],duolingo
2,honey,['miód'],[],duolingo
4,unhappy,['nieszczęśliwy'],[],duolingo
5,relaxing,"['relaksujący', 'odprężający']",[],duolingo
